In [1]:
import duckdb
import json

In [6]:
FILE_PATH = "../local-data/ED-2025-OPE-0944_2026-03-30.json"
ZIP_FILE_PATH = "../local-data/ED-2025-OPE-0944_2026-03-30.json.gz"

In [7]:
with open(FILE_PATH) as f:
    data = json.load(f)

In [12]:
data.keys()

dict_keys(['docket', 'documents', 'comments'])

In [13]:
data['docket'].keys()

dict_keys(['docket_id', 'agency_code', 'title', 'docket_type', 'modify_date', 'abstract'])

In [14]:
data['docket']

{'docket_id': 'ED-2025-OPE-0944',
 'agency_code': 'ED',
 'title': 'Reimagining and Improving Student Education (RISE) Committee',
 'docket_type': 'Rulemaking',
 'modify_date': '2026-02-12T17:11:44Z',
 'abstract': 'The Secretary proposes to amend the regulations for the Federal student loan programs authorized under the Higher Education Act to implement changes made by the One Big Beautiful Bill Act signed into law by President Trump on July 4, 2025. \n\nThese proposed regulations would amend annual and aggregate loan limits for graduate, professional, and parent loan borrowers, simplify the existing myriad of repayment plans into two plans for new borrowers, sunset Income-Contingent Repayment plans, provide defaulted borrowers a second chance to rehabilitate their loans and resume repayment, and make conforming amendments for consolidation loans, deferments, forbearances.'}

In [15]:
data['documents']

[{'document_id': 'ED-2025-OPE-0944-0002',
  'title': 'Information Collection Requirement: Supporting Statement',
  'document_type': 'Supporting & Related Material',
  'posted_date': '2026-01-30T05:00:00Z',
  'comment_start_date': None,
  'comment_end_date': None,
  'file_url': None},
 {'document_id': 'ED-2025-OPE-0944-0001',
  'title': 'Reimagining and Improving Student Education',
  'document_type': 'Proposed Rule',
  'posted_date': '2026-01-30T05:00:00Z',
  'comment_start_date': '2026-01-30T05:00:00Z',
  'comment_end_date': '2026-03-03T04:59:59Z',
  'file_url': 'https://downloads.regulations.gov/ED-2025-OPE-0944-0001/content.pdf'},
 {'document_id': 'ED-2025-OPE-0944-0003',
  'title': 'Information Collection Requirement: Forms and Instruments',
  'document_type': 'Supporting & Related Material',
  'posted_date': '2026-01-30T05:00:00Z',
  'comment_start_date': None,
  'comment_end_date': None,
  'file_url': None}]

In [33]:
with open("local-data/comments-only.json", 'w') as f:
    f.write(json.dumps(data['comments']))

In [17]:
len(data['comments'])

17730

In [20]:
# TODOs
# - get the comments themselves into duckdb for querying, not the whole json
# - comment text has html
# - need to drop "see attached file"
# - how do I extract an organization from the plain text?
# - do i just code the individuals (I am an art therapist) as "individual?" or "no affiliation available"

In [30]:
data['comments'][10]

{'comment_id': 'ED-2025-OPE-0944-17455',
 'docket_id': 'ED-2025-OPE-0944',
 'agency_code': 'ED',
 'title': 'Comment on FR Doc # 2026-01912',
 'comment': 'As a professional social worker, I am writing to express my deep concerns about the Department of Education&rsquo;s proposed rule categorizing social work as a graduate degree rather than a professional degree. This reclassification would limit the amount of federal student loans available to social work students, forcing many to rely on private loans that lack the same borrower protections and are not eligible for the Public Service Loan Forgiveness Program.<br/><br/>Without the federal student loans I received, I would never have been able to become the professional social worker I am today. I come from a disadvantaged community myself. Higher education was not easily accessible or financially feasible without federal support. Those loans were not simply financial assistance; they were an investment that allowed me to discover my pu

In [34]:
COMMENTS_PATH = "local-data/comments-only.json"

In [35]:
stmt = f"""
CREATE TABLE comments AS
    SELECT *
    FROM read_json('{COMMENTS_PATH}');
"""
duckdb.sql(stmt)

In [37]:
duckdb.sql("SELECT posted_date, modify_date, receive_date FROM comments LIMIT 10;")

┌─────────────────────┬─────────────────────┬─────────────────────┐
│     posted_date     │     modify_date     │    receive_date     │
│      timestamp      │      timestamp      │      timestamp      │
├─────────────────────┼─────────────────────┼─────────────────────┤
│ 2026-03-04 05:00:00 │ 2026-03-04 16:22:41 │ 2026-03-02 05:00:00 │
│ 2026-03-04 05:00:00 │ 2026-03-04 16:22:46 │ 2026-03-02 05:00:00 │
│ 2026-03-04 05:00:00 │ 2026-03-04 16:22:50 │ 2026-03-02 05:00:00 │
│ 2026-03-04 05:00:00 │ 2026-03-04 16:23:09 │ 2026-03-02 05:00:00 │
│ 2026-03-04 05:00:00 │ 2026-03-04 16:23:14 │ 2026-03-02 05:00:00 │
│ 2026-03-04 05:00:00 │ 2026-03-04 16:23:37 │ 2026-03-02 05:00:00 │
│ 2026-03-04 05:00:00 │ 2026-03-04 16:24:16 │ 2026-03-02 05:00:00 │
│ 2026-03-04 05:00:00 │ 2026-03-04 16:24:27 │ 2026-03-02 05:00:00 │
│ 2026-03-04 05:00:00 │ 2026-03-04 16:24:31 │ 2026-03-02 05:00:00 │
│ 2026-03-04 05:00:00 │ 2026-03-04 16:24:36 │ 2026-03-02 05:00:00 │
├─────────────────────┴─────────────────────┴───

In [39]:
duckdb.sql("""
SELECT 
    MIN(posted_date) AS min_posted_date,
    MIN(modify_date) AS min_modify_date,
    MIN(receive_date) AS min_receive_date 
FROM comments;
""")

┌─────────────────────┬─────────────────────┬─────────────────────┐
│   min_posted_date   │   min_modify_date   │  min_receive_date   │
│      timestamp      │      timestamp      │      timestamp      │
├─────────────────────┼─────────────────────┼─────────────────────┤
│ 2026-02-02 05:00:00 │ 2026-02-02 18:40:12 │ 2026-01-30 05:00:00 │
└─────────────────────┴─────────────────────┴─────────────────────┘

In [40]:
duckdb.sql("""
SELECT 
    MAX(posted_date) AS max_posted_date,
    MAX(modify_date) AS max_modify_date,
    MAX(receive_date) AS max_receive_date 
FROM comments;
""")

┌─────────────────────┬─────────────────────┬─────────────────────┐
│   max_posted_date   │   max_modify_date   │  max_receive_date   │
│      timestamp      │      timestamp      │      timestamp      │
├─────────────────────┼─────────────────────┼─────────────────────┤
│ 2026-03-04 05:00:00 │ 2026-03-04 17:29:59 │ 2026-03-02 05:00:00 │
└─────────────────────┴─────────────────────┴─────────────────────┘

In [45]:
df = duckdb.sql("""
SELECT receive_date, COUNT(1) FROM comments GROUP BY receive_date ORDER BY receive_date;
""")

┌─────────────────────┬──────────┐
│    receive_date     │ count(1) │
│      timestamp      │  int64   │
├─────────────────────┼──────────┤
│ 2026-01-30 05:00:00 │      279 │
│ 2026-01-31 05:00:00 │      201 │
│ 2026-02-01 05:00:00 │      179 │
│ 2026-02-02 05:00:00 │      540 │
│ 2026-02-03 05:00:00 │      563 │
│ 2026-02-04 05:00:00 │      409 │
│ 2026-02-05 05:00:00 │      536 │
│ 2026-02-06 05:00:00 │      362 │
│ 2026-02-07 05:00:00 │      175 │
│ 2026-02-08 05:00:00 │      190 │
│          ·          │       ·  │
│          ·          │       ·  │
│          ·          │       ·  │
│ 2026-02-22 05:00:00 │      194 │
│ 2026-02-23 05:00:00 │      787 │
│ 2026-02-24 05:00:00 │     1226 │
│ 2026-02-25 05:00:00 │     1177 │
│ 2026-02-26 05:00:00 │     1027 │
│ 2026-02-27 05:00:00 │      919 │
│ 2026-02-28 05:00:00 │      542 │
│ 2026-03-01 05:00:00 │     1037 │
│ 2026-03-02 05:00:00 │     1733 │
│ NULL                │        1 │
├─────────────────────┴──────────┤
│ 33 rows (20 shown)

In [62]:
# find repeated text
duckdb.sql("""
    SELECT COUNT(DISTINCT comment) as comment_count
    FROM comments
    --GROUP BY comment
    ORDER BY comment_count DESC
    LIMIT 20
""")

┌───────────────┐
│ comment_count │
│     int64     │
├───────────────┤
│         17196 │
└───────────────┘

In [78]:
# look at the dupes 
duckdb.sql("""
WITH flagged AS (
    SELECT comment,
        COUNT(1) OVER (PARTITION BY comment) > 3 as dupe_flag
        FROM comments
        WHERE LOWER(comment) NOT LIKE 'see attached file(s)' AND comment != 'See attached file' AND comment != 'See attached file.'
) SELECT comment, COUNT(*) as comment_ct FROM flagged WHERE dupe_flag GROUP BY comment ORDER BY comment_ct DESC LIMIT 10;
""")

┌───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────